In [ ]:
# Meesho CoD Trust Analysis
# Author: Vishal | IIT Madras BS Data Science
# Goal: Investigate root cause of Meesho's Cash on Delivery dominance
# Data: 133,300 self-scraped Google Play Store reviews

In [ ]:
# ── SECTION 1: SCRAPE REVIEWS ──────────────────────────────────────────────
# Scraping Meesho's Play Store reviews sorted newest first
# target_date sets how far back we go
# time.sleep(2) prevents Google from rate-limiting the scraper
 
from google_play_scraper import reviews, Sort
import pandas as pd
import time
import datetime

all_reviews = []
token = None
target_date = datetime.datetime(2024, 4, 1)

while True:
    batch, token = reviews(
        'com.meesho.supply',
        lang='en',
        country='in',
        sort=Sort.NEWEST,
        count=100,
        continuation_token=token
    )
    all_reviews.extend(batch)
    
    oldest = min(r['at'] for r in batch)
    print(f"Collected: {len(all_reviews)} | Oldest in batch: {oldest}")
    
    if oldest < target_date:
        break
    
    time.sleep(2)

df = pd.DataFrame(all_reviews)
df_filtered = df[df['score'] <= 2]
df_filtered.to_csv('meesho_reviews.csv', index=False)
print(f"Done. Total 1-2 star reviews saved: {len(df_filtered)}")

Collected: 100 | Oldest in batch: 2026-05-21 09:37:20
Collected: 200 | Oldest in batch: 2026-05-21 01:03:36
Collected: 300 | Oldest in batch: 2026-05-20 21:32:00
Collected: 400 | Oldest in batch: 2026-05-20 19:17:30
Collected: 500 | Oldest in batch: 2026-05-20 16:28:58
Collected: 600 | Oldest in batch: 2026-05-20 13:40:32
Collected: 700 | Oldest in batch: 2026-05-20 11:29:45
Collected: 800 | Oldest in batch: 2026-05-20 08:28:31
Collected: 900 | Oldest in batch: 2026-05-19 23:40:14
Collected: 1000 | Oldest in batch: 2026-05-19 20:56:03
Collected: 1100 | Oldest in batch: 2026-05-19 18:43:12
Collected: 1200 | Oldest in batch: 2026-05-19 16:08:10
Collected: 1300 | Oldest in batch: 2026-05-19 13:33:22
Collected: 1400 | Oldest in batch: 2026-05-19 10:55:18
Collected: 1500 | Oldest in batch: 2026-05-19 08:34:23
Collected: 1600 | Oldest in batch: 2026-05-18 23:19:48
Collected: 1700 | Oldest in batch: 2026-05-18 20:52:39
Collected: 1800 | Oldest in batch: 2026-05-18 18:05:27
Collected: 1900 | O

ValueError: min() iterable argument is empty

In [ ]:
# ── SECTION 2: BUILD AND SAVE DATAFRAME ──────────────────────────────────

df = pd.DataFrame(all_reviews)
df.to_csv('meesho_all_reviews_partial.csv', index=False)
print(f"Saved {len(df)} reviews")
print(f"Date range: {df['at'].min()} to {df['at'].max()}")

Saved 133300 reviews
Date range: 2025-10-23 21:32:28 to 2026-05-21 11:39:05


In [44]:
print(df['at'].min())
print(df['at'].max())

2025-10-23 21:32:28
2026-05-21 11:39:05


In [ ]:
# ── SECTION 3: FILTER NEGATIVE REVIEWS ────────────────────────────────────
# Keep only 1 and 2 star reviews — complaint signal lives here

df_filtered = df[df['score'] <= 2]
print(df_filtered['at'].min())
print(df_filtered['at'].max())
print(len(df_filtered))

2025-10-23 21:32:28
2026-05-21 11:17:14
23149


In [ ]:
# ── SECTION 4: MONTHLY TREND ───────────────────────────────────────────────
# Identify complaint volume spikes month by month

df_filtered['month'] = df_filtered['at'].dt.to_period('M')
monthly_counts = df_filtered.groupby('month').size()
print(monthly_counts)

month
2025-10    1370
2025-11    3486
2025-12    3318
2026-01    3110
2026-02    2825
2026-03    3500
2026-04    3248
2026-05    2292
Freq: M, dtype: int64


In [ ]:
# ── SECTION 5: KEYWORD ANALYSIS ───────────────────────────────────────────
# Map complaint categories across all negative reviews
# Financial keywords combined = 23.5% — second highest after delivery

keywords = ['payment', 'cod', 'refund', 'money', 'deliver', 
            'return', 'fraud', 'fake', 'support', 'cancel']

for keyword in keywords:
    count = df_filtered['content'].str.contains(keyword, case=False, na=False).sum()
    print(f"{keyword}: {count}")

payment: 891
cod: 336
refund: 1777
money: 1418
deliver: 6869
return: 2760
fraud: 1356
fake: 578
support: 1487
cancel: 2668


In [54]:
keywords = ['payment', 'refund', 'money', 'cod', 'deliver', 
            'return', 'cancel', 'support', 'fake', 'fraud', 'trust']

for keyword in keywords:
    count = df_filtered['content'].str.contains(keyword, case=False, na=False).sum()
    pct = round(count/len(df_filtered)*100, 1)
    print(f"{keyword}: {count} reviews ({pct}%)")

KeyboardInterrupt: 

In [52]:
fraud_reviews = df_filtered[
    df_filtered['content'].str.contains('fraud', case=False, na=False)
]
print(fraud_reviews['content'].head(50).tolist())

['A very bad online shopping platform I recommend not installing it or making any purchases through it.This is a fraudulent app and it does not provide any responsibility.', "very worst aap don't download it never because it is a fraud aap this aap first take your order and give you a delivery date and than the delivery date is becoming near they cancelled your order this is happened without me two times so I request everyone don't download this froud application", 'yah company fraud se kuch na khariden yah company Paisa refund nahin karti hai Inka delivery boy hamara Paisa lekar bhag gaya hai', 'fraud', 'IT IS FRAUD APP, I have order kizoop unique study table on 30 march 2026 and payment done with credit card, and cancel same day due to long delivery time, Meesho not Initated refund rupees 5032 till date 20 MAY 2026. I have call many times to customer care but no response from the customer support team. all time customer support team ask me ( backend team work on your complaint and ca

In [ ]:
# ── SECTION 6: FRAUD + PAYMENT INTERSECTION ───────────────────────────────
# Isolate reviews where fraud complaint is specifically about money
# not product quality — direct evidence of payment trust failure

payment_fraud = df_filtered[
    df_filtered['content'].str.contains('fraud', case=False, na=False) &
    df_filtered['content'].str.contains('payment|refund|money|paise|deduct|debit', case=False, na=False)
]

print(f"Total fraud reviews: {df_filtered['content'].str.contains('fraud', case=False, na=False).sum()}")
print(f"Fraud + payment/refund/money: {len(payment_fraud)}")
print()
print("--- Sample reviews ---")
for review in payment_fraud['content'].head(30).tolist():
    print(review)
    print()

Total fraud reviews: 1356
Fraud + payment/refund/money: 480

--- Sample reviews ---
yah company fraud se kuch na khariden yah company Paisa refund nahin karti hai Inka delivery boy hamara Paisa lekar bhag gaya hai

IT IS FRAUD APP, I have order kizoop unique study table on 30 march 2026 and payment done with credit card, and cancel same day due to long delivery time, Meesho not Initated refund rupees 5032 till date 20 MAY 2026. I have call many times to customer care but no response from the customer support team. all time customer support team ask me ( backend team work on your complaint and call back to you) but no call received, BEHANCHOD FUDDU GANDU APP HAI,, KOI SOLUTION NAHI MILA 30 MARCH SE

meesho ko main 1 star bhi dena nhi chahti qki maine is pr trust Kiya online payment karke suit order kiya (Vastrika Apparel company)ka tha damage, poor different product aya or return bhi nhi liya gya . hamara payment or trust dono hi gye. main other customers se yahi kehna chahti hu ki mees